In [1]:
from pyspark.sql.functions import count, col, when, broadcast, udf, pandas_udf, rand, monotonically_increasing_id
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, BooleanType, DoubleType, ArrayType, IntegerType, StructType, StructField
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
import datamol as dm
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import findspark
import hdbscan
import os
import sys

In [ ]:
spark.stop()

In [2]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("Molecule Analysis") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.network.timeout", "1000s") \
    .config("spark.executor.heartbeatInterval", "180s") \
    .config("spark.sql.broadcastTimeout", "1000s") \
    .config("spark.executor.waitTime", "1000s") \
    .getOrCreate()

In [3]:
filename = "leash-BELKA/train.parquet"
data = spark.read.parquet(filename)

data.printSchema()

data.show(5)

root
 |-- id: long (nullable = true)
 |-- buildingblock1_smiles: string (nullable = true)
 |-- buildingblock2_smiles: string (nullable = true)
 |-- buildingblock3_smiles: string (nullable = true)
 |-- molecule_smiles: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- binds: long (nullable = true)

+---+---------------------+---------------------+---------------------+--------------------+------------+-----+
| id|buildingblock1_smiles|buildingblock2_smiles|buildingblock3_smiles|     molecule_smiles|protein_name|binds|
+---+---------------------+---------------------+---------------------+--------------------+------------+-----+
|  0| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|        BRD4|    0|
|  1| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|         HSA|    0|
|  2| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|         sEH|    0|
|  3|

In [15]:
data.groupBy("binds").count().show()

+-----+---------+
|binds|    count|
+-----+---------+
|    0|293656924|
|    1|  1589906|
+-----+---------+



In [17]:
grouped_counts = data.groupBy("protein_name").agg(
    count(when(col('binds') == 1, 1)).alias('positive_count'),
    count(when(col('binds') == 0, 1)).alias('negative_count')
)

grouped_counts = grouped_counts.withColumn(
    "sampling_fraction",
    col("positive_count")/col("negative_count")
)

grouped_counts.show()

+------------+--------------+--------------+--------------------+
|protein_name|positive_count|negative_count|   sampling_fraction|
+------------+--------------+--------------+--------------------+
|         HSA|        408410|      98007200|0.004167142822160...|
|        BRD4|        456964|      97958646|0.004664866437618992|
|         sEH|        724532|      97691078| 0.00741656264659092|
+------------+--------------+--------------+--------------------+



In [19]:
sampling_fraction = broadcast(grouped_counts.select("protein_name", "sampling_fraction"))

data_with_fraction = data.join(sampling_fraction, on = "protein_name", how = "left")

downsampled_data = data_with_fraction.filter(col("binds") == 1).union(
    data_with_fraction.filter(col("binds") == 0).sampleBy(
        col("protein_name"), fractions = dict(sampling_fraction.rdd.collectAsMap()), seed = 42
    )
)

In [21]:
downsampled_data.groupby("binds").count().show()

+-----+-------+
|binds|  count|
+-----+-------+
|    1|1589906|
|    0|1592766|
+-----+-------+



**SMART DOWNSAMPLING**

*Based on descriptors*

*Desctiptors extraction*

In [4]:
data = data.filter(col("protein_name") == "sEH")

In [5]:
sampled_data = data.orderBy(rand()).limit(3000)

In [6]:
del data

Descriptors calculations

In [7]:
desc_schema = StructType([
    StructField("mol_wt", DoubleType(), True),
    StructField("mol_logp", DoubleType(), True),
    StructField("tpsa", DoubleType(), True),
]
)

@pandas_udf(desc_schema)
def calculate_desc(smiles_series: pd.Series) -> pd.DataFrame:
    results = []

    for smiles in smiles_series:
        mol = Chem.MolFromSmiles(smiles)

        if mol:
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            tpsa = Descriptors.TPSA(mol)           

        else:
            mw, logp, tpsa= [None] * 9

        results.append((mw, logp, tpsa))

    return pd.DataFrame(results, columns=desc_schema.fieldNames())

In [8]:
sampled_data_with_desc = sampled_data.withColumn("molecule", calculate_desc(col("molecule_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block1", calculate_desc(col("buildingblock1_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block2", calculate_desc(col("buildingblock2_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block3", calculate_desc(col("buildingblock3_smiles")))

In [9]:
selected_columns = [
    "id",
    col("molecule.mol_wt").alias("molecule_mol_wt"),
    col("molecule.mol_logp").alias("molecule_mol_logp"),
    col("molecule.tpsa").alias("molecule_tpsa"),
    col("block1.mol_wt").alias("block1_mol_wt"),
    col("block1.mol_logp").alias("block1_mol_logp"),
    col("block1.tpsa").alias("block1_tpsa"),
    col("block2.mol_wt").alias("block2_mol_wt"),
    col("block2.mol_logp").alias("block2_mol_logp"),
    col("block2.tpsa").alias("block2_tpsa"),
    col("block3.mol_wt").alias("block3_mol_wt"),
    col("block3.mol_logp").alias("block3_mol_logp"),
    col("block3.tpsa").alias("block3_tpsa"),
]

flattened_data = sampled_data_with_desc.select(*selected_columns)

In [10]:
del sampled_data_with_desc

In [11]:
desc_columns = [col_name for col_name in flattened_data.columns if col_name != "id"]

assembler = VectorAssembler(inputCols=desc_columns, outputCol="features")
sampled_data_with_features = assembler.transform(flattened_data)

In [12]:
del flattened_data

In [13]:
sampled_data_with_features = sampled_data_with_features.repartition(100)

In [14]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(sampled_data_with_features)
scaled_data = scaler_model.transform(sampled_data_with_features)

In [15]:
del sampled_data_with_features

In [16]:
scaled_data = scaled_data.repartition(200)

In [17]:
kmeans = KMeans(k=5, seed=42, featuresCol="scaled_features", predictionCol="kmeans_cluster")
kmeans_model = kmeans.fit(scaled_data)
clustered_data = kmeans_model.transform(scaled_data)

In [18]:
clustered_data.groupBy("kmeans_cluster").count().show()

+--------------+-----+
|kmeans_cluster|count|
+--------------+-----+
|             1|  758|
|             3|  928|
|             4|  469|
|             2|  468|
|             0|  377|
+--------------+-----+



In [ ]:
clustered_data.select("id", "kmeans_cluster").write.csv("kmeans_results.csv", header=True)